In [0]:
%python

## IMPORTS 
import pyspark.sql
from pyspark.sql import functions as F
from pyspark.sql.functions import col, current_timestamp, when, upper, initcap, trim
from delta.tables import DeltaTable

'''
 BRONCE table to dataframe
'''

df_table_bronce_logs = spark.table("banking_bronze.payment_gateway_logs")

df_table_bronce_logs.printSchema()
# display(df_table_bronce_logs)


'''
    1. DATA CASTING -> Data types (long, timestamp) come correctly typed from Bronze
'''


'''
    2. TEXT FORMATTING
'''

# 1. GATEWAY_NAME -> Uppercase and trimmed of spaces (e.g. "STRIPE", "PAYPAL", "ADYEN")
# 2. GATEWAY_STATUS -> Uppercase (e.g. "SUCCESS", "FAILED", "TIMEOUT")
# 3. DEVICE_TYPE -> Title Case (e.g. "Mobile", "Desktop", "POS")
# 4. GEO_LOCATION -> Uppercase for country/zone codes or Initcap if it is a city/country name

df_txt_format_logs = (
    df_table_bronce_logs
    .withColumn("gateway_name", upper(trim(col("gateway_name"))))
    .withColumn("gateway_status", upper(trim(col("gateway_status"))))
    .withColumn("device_type", initcap(trim(col("device_type"))))
    .withColumn("geo_location", when(F.length(trim(col("geo_location"))) <= 3, upper(trim(col("geo_location"))))
        .otherwise(initcap(trim(col("geo_location")))))
)


'''
 3. NULL HANDLING
'''

# 1. Keep NULL in txn_id (FK) in case there are orphan logs from payment attempts not registered as a transaction
# 2. Imputation of empty descriptive text columns with "NOT_PROVIDED"

for colum in ["gateway_name", "gateway_status", "device_type", "geo_location"]:
    df_txt_format_logs = df_txt_format_logs.withColumn(
        colum,
        when(col(colum).isNull(), "NOT_PROVIDED").otherwise(col(colum))
    )

df_logs_not_null = df_txt_format_logs
df_logs_not_null.show(4)


'''
    4. REMOVE DUPLICATES
'''

# 1. Deduplication by business combination:
# Assumes a single log entry per transaction, gateway, and processed timestamp
df_logs_dedupli = (
    df_logs_not_null
    .orderBy(col("processed_timestamp").desc())
    .dropDuplicates(["txn_id", "gateway_name", "processed_timestamp"])
)

print("Cantidad tras la DEDUPLICACIÓN:", df_logs_dedupli.count())
df_logs_dedupli.show(5)


'''
 5. BUSINESS LOGIC AND QUALITY VALIDATION
'''

# Quality checks for Silver auditing:
# 1. is_orphan_log: Flags if the log does not have an associated txn_id (technical failure payment attempt).
# 2. is_latency_anomaly: Detects atypical/negative latencies or inconsistencies (e.g., processing time <= 0 ms or null).
# 3. is_response_error: Flags if the gateway returned an HTTP/Gateway error code (e.g., codes >= 400 or NULL).

df_negocio_logs = (
    df_logs_dedupli
    .withColumn(
        "is_orphan_log",
        when(col("txn_id").isNull(), True).otherwise(False)
    )
    .withColumn(
        "is_latency_anomaly",
        when((col("processing_time_ms").isNull()) | (col("processing_time_ms") <= 0), True).otherwise(False)
    )
    .withColumn(
        "is_response_error",
        when((col("response_code").isNull()) | (col("response_code") >= 400), True).otherwise(False)
    )
)


'''
    6. AUDITING
'''

df_logs_auditoria = (
    df_negocio_logs
    .withColumn("created_at", current_timestamp())
    .withColumn("updated_at", current_timestamp())
    .withColumn("processed_at", current_timestamp())
)

df_logs_auditoria.show(4)


'''
    Write the new dataframe to the SILVER table
'''

(
    df_logs_auditoria.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("banking_silver.payment_gateway_logs")
)

silver_logs_table = spark.table("banking_silver.payment_gateway_logs")
display(silver_logs_table)

In [0]:
%python

## IMPORTS 
import pyspark.sql
from pyspark.sql import functions as F
from pyspark.sql.functions import col, current_timestamp, when, upper, trim
from delta.tables import DeltaTable

'''
    BRONZE table to dataframe
'''

df_table_bronce_bureau = spark.table("banking_bronze.credit_bureau_reports")

df_table_bronce_bureau.printSchema()
# display(df_table_bronce_bureau)


'''
 1. DATA CASTING -> Data types (long, date) come correctly typed from Bronze
'''


'''
 2. TEXT FORMATTING
'''

# 1. RISK_GRADE -> Standardization to uppercase and space trimming (e.g., "A", "B", "C", "HIGH", "LOW")
df_txt_format_bureau = df_table_bronce_bureau.withColumn(
    "risk_grade", 
    upper(trim(col("risk_grade")))
)


'''
 3. NULL HANDLING
'''

# 1. Check that there are no null customer_ids -> FILTER (Primary Foreign Key / Report Identifier)
print("Initial count:", df_txt_format_bureau.count())
df_bureau_null_filter = df_txt_format_bureau.filter(col("customer_id").isNotNull())
print("Count after filtering null customer_id:", df_bureau_null_filter.count())

df_bureau_not_null = df_bureau_null_filter

# Imputation of null descriptive columns with "NOT_PROVIDED"
df_bureau_not_null = df_bureau_not_null.withColumn(
    "risk_grade",
    when(col("risk_grade").isNull(), "NOT_PROVIDED").otherwise(col("risk_grade"))
)

#For external loans and overdue amounts, if they come as NULL, impute them to 0 to avoid breaking quantitative analyses
df_bureau_not_null = df_bureau_not_null \
    .withColumn("external_active_loans", when(col("external_active_loans").isNull(), 0).otherwise(col("external_active_loans"))) \
    .withColumn("external_overdue_amount", when(col("external_overdue_amount").isNull(), 0).otherwise(col("external_overdue_amount")))

df_bureau_not_null.show(4)


'''
 4. REMOVE DUPLICATES
'''

# 1. Deduplication by business combination: A customer should only have ONE report per pull date (bureau_pull_date).
# Keep the most recent record in case of multiple pulls on the same day.

df_bureau_dedupli = (
    df_bureau_not_null
    .orderBy(col("bureau_pull_date").desc())
    .dropDuplicates(["customer_id", "bureau_pull_date"])
)

print("Count after DEDUPLICATION:", df_bureau_dedupli.count())
df_bureau_dedupli.show(5)


'''
 5. BUSINESS LOGIC AND QUALITY VALIDATION
'''

# Quality controls for the Silver layer:
# 1. is_credit_score_invalid: The Credit Score is usually in the [300 - 850] range. Flag if null or out of technical range.
# 2. is_inconsistent_overdue: Detects if an overdue amount is recorded while recording 0 active loans.

df_negocio_bureau = (
    df_bureau_dedupli
    .withColumn(
        "is_credit_score_invalid",
        when((col("credit_score").isNull()) | (col("credit_score") < 300) | (col("credit_score") > 850), True)
        .otherwise(False)
    )
    .withColumn(
        "is_inconsistent_overdue",
        when((col("external_overdue_amount") > 0) & (col("external_active_loans") == 0), True)
        .otherwise(False)
    )
)


'''
 6. AUDITING
'''

# Gestión de marcas de tiempo de linaje técnico
df_bureau_auditoria = (
    df_negocio_bureau
    .withColumn("created_at", current_timestamp())
    .withColumn("updated_at", current_timestamp())
    .withColumn("processed_at", current_timestamp())
)

df_bureau_auditoria.show(4)


'''
 Write the new dataframe to the SILVER table
'''

(
    df_bureau_auditoria.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("banking_silver.credit_bureau_reports")
)

silver_bureau_table = spark.table("banking_silver.credit_bureau_reports")
display(silver_bureau_table)

In [0]:
%python

## IMPORTS
import pyspark.sql
from pyspark.sql import functions as F
from pyspark.sql.functions import col, current_timestamp, when, upper, trim
from delta.tables import DeltaTable

'''
BRONZE table to dataframe
'''

df_table_bronce_transactions = spark.table("banking_bronze.transactions")

df_table_bronce_transactions.printSchema()
# display(df_table_bronce_transactions)


'''
 1. DATA CASTING -> Data types (long, decimal, timestamp) come correctly typed from Bronze
'''


'''
 2. TEXT FORMATTING
'''

# 1. TXN_TYPE -> Uppercase and trimmed of spaces (e.g. "TRANSFER", "DEPOSIT", "WITHDRAWAL")
# 2. CHANNEL -> Uppercase (e.g. "ATM", "ONLINE", "BRANCH")
# 3. STATUS -> Uppercase or Initcap depending on project standards (e.g. "COMPLETED", "FAILED", "PENDING")

df_txt_format_txn = (
    df_table_bronce_transactions
    .withColumn("txn_type", upper(trim(col("txn_type"))))
    .withColumn("channel", upper(trim(col("channel"))))
    .withColumn("status", upper(trim(col("status"))))
)


'''
 3. NULL HANDLING
'''

# 1. Compruebo que no hay ningún txn_id nulo -> FILTRO (Clave Primaria)
print("Initial count:", df_txt_format_txn.count())
df_txn_null_filter = df_txt_format_txn.filter(col("txn_id").isNotNull())
print("Count after filtering null PK:", df_txn_null_filter.count())

df_txn_not_null = df_txn_null_filter

# Imputation of empty descriptive columns
for colum in ["txn_type", "channel", "status"]:
    df_txn_not_null = df_txn_not_null.withColumn(
        colum,
        when(col(colum).isNull(), "NOT_PROVIDED").otherwise(col(colum))
    )

df_txn_not_null.show(4)


'''
 4. REMOVE DUPLICATES
'''

# 1. Deduplication by Primary Key (txn_id)
# Ordered by txn_timestamp desc (or created_at) to keep the latest transaction status 
df_txn_no_dupli = (
    df_txn_not_null
    .orderBy(col("txn_timestamp").desc(), col("created_at").desc())
    .dropDuplicates(["txn_id"])
)

# 2. Deduplication by business combination to detect duplicate transactions due to source system error:
# Same account, same type, same amount, and at the exact same timestamp
df_txn_dedupli = (
    df_txn_no_dupli
    .sort(col("created_at").desc())
    .dropDuplicates(["account_id", "txn_type", "amount", "txn_timestamp"])
)

print("Count after DEDUPLICATION:", df_txn_dedupli.count())
df_txn_dedupli.show(5)


'''
 5. BUSINESS LOGIC AND QUALITY VALIDATION
'''

# Quality controls for analytical auditing in Silver:
# 1. is_amount_invalid: Detects if transaction has no amount (NULL) or is <= 0 (except specific adjustments).
# 2. is_orphan_account: Flags if transaction does not have an associated account_id.

df_negocio_txn = (
    df_txn_dedupli
    .withColumn(
        "is_amount_invalid",
        when((col("amount").isNull()) | (col("amount") <= 0), True).otherwise(False)
    )
    .withColumn(
        "is_orphan_account",
        when(col("account_id").isNull(), True).otherwise(False)
    )
)


'''
 6. AUDITING
'''

df_txn_auditoria = (
    df_negocio_txn
    .withColumn("created_at", F.coalesce(col("created_at"), current_timestamp()))
    .withColumn("updated_at", current_timestamp())
    .withColumn("processed_at", current_timestamp())
)

df_txn_auditoria.show(4)


'''
 Write the new dataframe to the SILVER table
'''

(
    df_txn_auditoria.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("banking_silver.transactions")
)

silver_txn_table = spark.table("banking_silver.transactions")
display(silver_txn_table)

In [0]:
%python

## IMPORTS 
import pyspark.sql
from pyspark.sql import functions as F
from pyspark.sql.functions import col, current_timestamp, when, initcap, upper, trim
from delta.tables import DeltaTable

'''
 BRONZE table to dataframe
'''

df_table_branches = spark.table("banking_bronze.branches")

df_table_branches.printSchema()
# display(df_table_branches)


'''
 1. DATA CASTING -> Data types are already correctly defined in Bronze
'''


'''
 2. TEXT FORMAT
'''

# 1. BRANCH_CODE -> Uppercase code without spaces
# 2. BRANCH_NAME -> Title Case (Initcap)
# 3. CITY, STATE, REGION -> Title Case (Initcap)

df_txt_format_branch = (
    df_table_branches
    .withColumn("branch_code", upper(trim(col("branch_code"))))
    .withColumn("branch_name", initcap(trim(col("branch_name"))))
    .withColumn("city", initcap(trim(col("city"))))
    .withColumn("state", initcap(trim(col("state"))))
    .withColumn("region", upper(trim(col("region"))))
)


'''
 3. NULL HANDLING
'''

# 1. Check that there are no null branch_codes -> FILTER (Primary Key)
print("Cantidad inicial:", df_txt_format_branch.count())
df_branches_null_filter = df_txt_format_branch.filter(col("branch_code").isNotNull())
print("Cantidad tras filtrar PK nula:", df_branches_null_filter.count())

df_branches_not_null = df_branches_null_filter

# Imputation of null descriptive text columns with "NOT_PROVIDED"
for colum in ["branch_name", "city", "state", "region"]:
    df_branches_not_null = df_branches_not_null.withColumn(
        colum,
        when(col(colum).isNull(), "NOT_PROVIDED").otherwise(col(colum))
    )

df_branches_not_null.show(4)


'''
    4. REMOVE DUPLICATES
'''

# 1. Remove duplicate records by PK (branch_code) keeping the most recent record
# Note: If updated_at does not exist in Bronze, created_at is used as the ordering criterion.
df_branch_no_dupli = (
    df_branches_not_null
    .orderBy(col("created_at").desc())
    .dropDuplicates(["branch_code"])
)

# 2. Deduplication by business combination (Same branch name in the same city and state)
df_branch_dedupli = (
    df_branch_no_dupli
    .sort(col("created_at").desc())
    .dropDuplicates(["branch_name", "city", "state"])
)

print("Count after DEDUPLICATION:", df_branch_dedupli.count())
df_branch_dedupli.show(5)


'''
5. BUSINESS LOGIC AND VALIDATION
'''

# Quality audit flag: Detects if the branch does not have an assigned region or valid location
df_negocio_branch = df_branch_dedupli.withColumn(
    "is_location_incomplete",
    when((col("city") == "NOT_PROVIDED") | (col("state") == "NOT_PROVIDED") | (col("region") == "NOT_PROVIDED"), True)
    .otherwise(False)
)


'''
 6. AUDITING
'''

df_branch_auditoria = (
    df_negocio_branch
    .withColumn("created_at", F.coalesce(col("created_at"), current_timestamp()))
    .withColumn("updated_at", current_timestamp())
    .withColumn("processed_at", current_timestamp())
)

df_branch_auditoria.show(4)


'''
 Write the new dataframe to the SILVER table
'''

(
    df_branch_auditoria.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("banking_silver.branches")
)

silver_branch_table = spark.table("banking_silver.branches")
display(silver_branch_table)

In [0]:
%python

## IMPORTS 
import pyspark.sql
from pyspark.sql import functions as F
from pyspark.sql.functions import col, current_timestamp, when, initcap, upper, trim
from delta.tables import DeltaTable

'''
 BRONZE table to dataframe
'''

df_table_accounts = spark.table("banking_bronze.accounts")

df_table_accounts.printSchema()
# display(df_table_accounts)


'''
 1. DATA CASTING -> In this case, types are already filtered/correct
'''


'''
 2. TEXT FORMATTING
'''

# 1. ACCOUNT_TYPE -> Title Case (Initcap), and WHITELIST [Savings or Current]
# 2. CURRENCY -> ALL Uppercase
# 3. BRANCH CODE -> Worth using a regexp rlike(pattern) -> depends on another table, left unchanged for now
# 4. STATUS -> ALL Uppercase (Note: requirement comment states ALL upper, adjusted code accordingly)

df_txt_format_acc = (
    df_table_accounts
    .withColumn(
        "account_type", 
        when(initcap(trim(col("account_type"))).isin(["Savings", "Current"]), initcap(trim(col("account_type"))))
        .otherwise("NOT_PROVIDED")
    )
    .withColumn("currency", upper(trim(col("currency"))))
    .withColumn("status", upper(trim(col("status"))))
)

# df_txt_format_acc.show(3)


'''
 3. NULL HANDLING
'''

# 1. Check that there are no null account_ids -> FILTER
print("Initial count:", df_txt_format_acc.count())
df_accounts_null_filter = df_txt_format_acc.filter(col("account_id").isNotNull())

print("Count after filtering null PK:", df_accounts_null_filter.count())
df_accounts_not_null = df_accounts_null_filter

# Modify nulls in columns that do not affect other tables (excluding branch_code and customer_id)
for colum in ["currency", "status"]:
    df_accounts_not_null = df_accounts_not_null.withColumn(
        colum,
        when(col(colum).isNull(), "NOT_PROVIDED").otherwise(col(colum))
    )

df_accounts_not_null.show(4)


'''
 4. REMOVE DUPLICATES
'''

# 1. Remove duplicate account_ids. Order by date first to keep the most recent record.
df_account_no_dupli = (
    df_accounts_not_null
    .orderBy(col("updated_at").desc(), col("created_at").desc())
    .dropDuplicates(["account_id"])
)

# 2. Remove accounts that are identical except for the account_id due to source system errors
df_account_dedupli = (
    df_account_no_dupli
    .sort(col("updated_at").desc())
    .dropDuplicates(["customer_id", "account_type", "branch_code", "opened_date"])
)

print("Count after DEDUPLICATION:", df_account_dedupli.count())
df_account_dedupli.show(5)


'''
 5. BUSINESS LOGIC AND VALIDATION
'''

# 1. SHOULD BALANCE BE POSITIVE OR 0? -> Yes. What matters is handling nulls.
# If null and active, flag it for audit so it doesn't affect Gold-layer metrics.

df_negocio_account = df_account_dedupli.withColumn(
    "is_balance_null", 
    when((col("balance").isNull()) & (col("status") == "ACTIVE"), True)
    .otherwise(False)
)


'''
 6. AUDITING
'''

df_account_auditoria = (
    df_negocio_account
    .withColumn("created_at", F.coalesce(col("created_at"), current_timestamp()))
    .withColumn("updated_at", current_timestamp())
    .withColumn("processed_at", current_timestamp())
)

df_account_auditoria.show(4)


'''
 Write the new dataframe to the SILVER table
'''

(
    df_account_auditoria.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("banking_silver.accounts")
)

silver_account_table = spark.table("banking_silver.accounts")
display(silver_account_table)

In [0]:
# MISCELLANEOUS IMPORTS

import pyspark.sql 
from pyspark.errors import PySparkException
import json
from pyspark.sql import functions as F
from pyspark.sql.functions import col, current_timestamp, trim, initcap, lower, upper, datediff, current_date, max as spark_max
from delta.tables import DeltaTable # Data Lake adds ACID transactions, UPDATE/DELETE/MERGE support, and time travel to Spark DataFrames stored as Parquet


# 1. Open Bronze table and create a DataFrame from it
# 2. Perform sequential transformations on this DataFrame:

# table_first = spark.table("banking.customers")
# display(table_first)
table_br_customers = spark.table("banking_bronze.customers")

## COLUMNS:::: ["customer_id", "first_name", "last_name", "date_of_birth", "pan_number", "email", "phone_number", "kyc_status", "branch_code", "created_at", "updated_at"]

table_br_customers.printSchema()
# display(table_br_customers) # In Google Colab it uses show(), not here. Everything else is identical.

'''
 1. Data type standardization (CASTING)
'''

### df_limpio = df_bronze.withColumn("customer_id", col("customer_id").cast("int")) \
    ### .withColumn("date_of_birth", col("date_of_birth").cast("date"))

'''
 2. Text Format Standardization -> Space removal, capitalization, etc.
'''
print("WORKS UP TO THIS POINT")
## No spaces, Title Case

email_pattern = r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"
phone_pattern = r"^[0-9]{10}$"

df_silver_customers = (
    table_br_customers
    .withColumn("first_name", initcap(trim(col("first_name"))))
    .withColumn("last_name", initcap(trim(col("last_name"))))
    .withColumn("email", F.when(lower(trim(col("email"))).rlike(email_pattern), lower(trim(col("email")))).otherwise("NOT_PROVIDED"))
    .withColumn("kyc_status", upper(trim(col("kyc_status"))))
    .withColumn("pan_number", upper(trim(col("pan_number"))))
    .withColumn("phone_number", F.when(col("phone_number").rlike(phone_pattern), col("phone_number")).otherwise("NOT_PROVIDED"))
)

# df_silver_customers.show(3)

# TO CONFIRM REGEXP --> rlike()




'''
 3. Null handling (NULL HANDLING)
'''
print("TOTAL WITH NULLS ", df_silver_customers.count())
# If ID is missing, storing this account in silver makes no sense
df_with_id = df_silver_customers.filter(col("customer_id").isNotNull())
print("TOTAL WITHOUT NULLS ", df_with_id.count())

# CHANGING TO A SPECIFIC VALUE --> FILLNA --> df.fillna({"city": "Unknown", "salary": 0})
df_half_null = df_with_id.fillna({"kyc_status": "...."})
df_not_null = df_half_null

# Updating a set of columns all to the same default...
for colum in ["first_name", "last_name", "pan_number", "phone_number"]:
    df_not_null = df_not_null.withColumn(
        colum, 
        F.when(col(colum).isNull(), "NOT_PROVIDED").otherwise(col(colum))
    )

# df_not_null.show(10)



'''
 4. Record deduplication (NO DUPLICATES)
'''
# 1. Remove ID DUPLICATES!
df_no_duplicate = df_not_null.dropDuplicates(["customer_id"])

# df_without_duplicates = df_not_null.dropDuplicates(["first_name", "last_name", "date_of_birth"]) 
# # This would keep the first record encountered and drop the rest. 

# df_without_duplicates = df_no_duplicate.distinct() --> Redundant, already handled by dropDuplicates --> REMOVES ROWS THAT ARE EXACTLY IDENTICAL ACROSS ALL COLUMNS

print(f"{df_no_duplicate.count()} is the count after REMOVING DUPLICATES")


'''
 5. Basic Business Logic VALIDATION and FILTERING
'''

# Minor validation: add a new status column for minors
df_maiores = df_no_duplicate.withColumn(
    "is_legal", 
    F.when((datediff(current_date(), col("date_of_birth")) < (18 * 365.25)), "MENOR").otherwise("MAYOR")
)

df_maiores.show(8)

print("ADULTS COUNT: ", df_maiores.filter(col("is_legal") == "MAYOR").count())

'''
 6. AUDITING: new column with date/time
'''

# Data modification timestamp -> updated_at

# Spark pipeline execution timestamp -> processed_